# 하이브리드 예측 모델: Chronos + iTransformer

| 모듈 | 역할 |
|------|------|
| **Chronos T5 Encoder** | Close price → 시계열 temporal 임베딩 (512-dim) |
| **iTransformer Variate Attention** | 20개 기술지표 × 60일 → 피처 간 상관 임베딩 |
| **Fusion MLP** | 두 임베딩 결합 → 수익률 예측 + 64-dim RL state |

실행 순서:
1. Chronos 임베딩 추출 (최초 1회, Drive에 캐시)
2. ChronosITransformer 학습
3. RL용 임베딩 추출 (parquet → rl_embeddings.h5)

> 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q chronos-forecasting einops h5py pyarrow tqdm

In [ ]:
import gc
import warnings
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.stats import spearmanr
from tqdm.auto import tqdm
from pathlib import Path

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['axes.unicode_minus'] = False

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

DRIVE_ROOT = Path('/content/drive/MyDrive/grad_project')
DATA_DIR   = DRIVE_ROOT / 'data'
MODEL_DIR  = DRIVE_ROOT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

H5_PATH    = DATA_DIR / 'sequences' / 'sequences.h5'
PARQUET    = DATA_DIR / 'kospi_valid.parquet'
CLUSTER_CSV= DATA_DIR / 'cluster_assignments.csv'
EMB_CACHE  = DATA_DIR / 'chronos_embs.npz'
RL_EMB_H5  = DATA_DIR / 'rl_embeddings.h5'

SEQ_LEN    = 60
N_PATCHES  = 12
PATCH_LEN  = 5
N_FEATURES = 20
PRED_LEN   = 5
CHRONOS_DIM= 512  # chronos-t5-small hidden dim

D_MODEL    = 128
N_HEADS    = 4
N_LAYERS   = 2
DROPOUT    = 0.1
BATCH_SIZE = 256
N_EPOCHS   = 15
LR         = 5e-4
PATIENCE   = 3
MAX_TRAIN  = 80_000

assert H5_PATH.exists(), f'sequences.h5 없음: {H5_PATH}'
print(f'sequences.h5: {H5_PATH.stat().st_size/1e9:.2f} GB')

## 1. 데이터 로드 (H5)

In [ ]:
class KospiH5Dataset(Dataset):
    def __init__(self, h5_path, split, max_samples=None):
        with h5py.File(h5_path, 'r') as f:
            n = f[split]['y_ret'].shape[0]
            sl = slice(0, n, n // max_samples) if max_samples and n > max_samples else slice(None)
            self.X_flat = f[split]['X_flat'][sl][:max_samples or n].astype(np.float32)
            self.y_ret  = f[split]['y_ret'][sl][:max_samples or n].astype(np.float32)
            self.y_dir  = f[split]['y_dir'][sl][:max_samples or n].astype(np.int64)
        print(f'{split:5s}: {len(self.y_ret):>8,}개 | 상승비율: {self.y_dir.mean():.3f}')

    def __len__(self):
        return len(self.y_ret)

    def __getitem__(self, i):
        return (
            torch.from_numpy(self.X_flat[i]),   # (60, 20)
            torch.tensor(self.y_ret[i]),
            torch.tensor(self.y_dir[i]),
        )


train_ds = KospiH5Dataset(H5_PATH, 'train', max_samples=MAX_TRAIN)
val_ds   = KospiH5Dataset(H5_PATH, 'val')
test_ds  = KospiH5Dataset(H5_PATH, 'test')

## 2. Chronos 임베딩 추출 (최초 1회 후 캐시)

In [ ]:
from chronos import ChronosPipeline

gc.collect()
torch.cuda.empty_cache()

pipeline = ChronosPipeline.from_pretrained(
    'amazon/chronos-t5-small',
    device_map=DEVICE,
    torch_dtype=torch.float32,
)
print('Chronos 로드 완료')


@torch.no_grad()
def extract_chronos_embs(close_prices_np, batch_size=256, desc=''):
    """close_prices_np: (N, seq_len) float32 → (N, CHRONOS_DIM) embeddings"""
    tokenizer = pipeline.tokenizer
    encoder   = pipeline.model.model.encoder
    all_embs  = []
    for i in tqdm(range(0, len(close_prices_np), batch_size), desc=desc or 'Chronos'):
        ctx  = torch.tensor(close_prices_np[i:i+batch_size], dtype=torch.float32).to(DEVICE)
        ids, mask, _ = tokenizer.context_input_transform(ctx)
        out  = encoder(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE))
        h    = out.last_hidden_state                          # (B, L, D)
        m    = mask.to(DEVICE).unsqueeze(-1).float()
        emb  = (h * m).sum(1) / m.sum(1).clamp(min=1)       # masked mean pool
        all_embs.append(emb.cpu().float().numpy())
    return np.concatenate(all_embs, axis=0)

In [ ]:
%%time
if EMB_CACHE.exists():
    cached  = np.load(EMB_CACHE)
    tr_emb  = cached['train']
    va_emb  = cached['val']
    te_emb  = cached['test']
    print(f'캐시 로드: train={tr_emb.shape}, val={va_emb.shape}, test={te_emb.shape}')
else:
    print('Chronos 임베딩 추출 (최초 1회)...')
    tr_emb = extract_chronos_embs(train_ds.X_flat[:, :, 0], desc='train')
    va_emb = extract_chronos_embs(val_ds.X_flat[:, :, 0],   desc='val')
    te_emb = extract_chronos_embs(test_ds.X_flat[:, :, 0],  desc='test')
    np.savez(EMB_CACHE, train=tr_emb, val=va_emb, test=te_emb)
    print(f'저장 완료: {EMB_CACHE}')

## 3. ChronosITransformer 모델 정의

In [ ]:
class ChronosITransformer(nn.Module):
    """
    Chronos T5 인코더(temporal) + iTransformer variate attention(cross-feature)
    두 표현을 fusion MLP로 결합해 수익률 예측 및 64-dim RL state 생성
    """
    def __init__(self, seq_len=60, n_features=20,
                 chronos_dim=512, d_model=128, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        # Chronos 임베딩 투영
        self.chronos_proj = nn.Linear(chronos_dim, d_model)
        self.chronos_norm = nn.LayerNorm(d_model)

        # iTransformer: 각 기술지표의 전체 시계열을 variate 토큰으로 처리
        self.variate_embed = nn.Linear(seq_len, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model, nhead, d_model * 4, dropout, batch_first=True, norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, n_layers)
        self.it_norm     = nn.LayerNorm(d_model)

        # Fusion: (d_model + n_features*d_model) → 64
        fusion_in = d_model + n_features * d_model
        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 64),        nn.GELU(),
        )
        self.head = nn.Linear(64, 1)

    def _it_features(self, x_flat):
        x = x_flat.permute(0, 2, 1)           # (B, N_feat, T)
        x = self.variate_embed(x)              # (B, N_feat, D)
        x = self.it_norm(self.transformer(x)) # (B, N_feat, D)
        return x.flatten(1)                    # (B, N_feat*D)

    def forward(self, x_flat, c_emb):
        chron = self.chronos_norm(self.chronos_proj(c_emb))  # (B, D)
        it    = self._it_features(x_flat)                    # (B, N*D)
        fused = self.fusion(torch.cat([chron, it], dim=-1))  # (B, 64)
        return self.head(fused).squeeze(-1)

    @torch.no_grad()
    def embed(self, x_flat, c_emb):
        """64-dim RL state representation"""
        self.eval()
        chron = self.chronos_norm(self.chronos_proj(c_emb))
        it    = self._it_features(x_flat)
        return self.fusion(torch.cat([chron, it], dim=-1))   # (B, 64)


model_info = ChronosITransformer(SEQ_LEN, N_FEATURES, CHRONOS_DIM, D_MODEL, N_HEADS, N_LAYERS)
n_params   = sum(p.numel() for p in model_info.parameters())
print(f'ChronosITransformer 파라미터: {n_params:,}')
del model_info

## 4. 하이브리드 데이터셋 & 로더

In [ ]:
class HybridDataset(Dataset):
    """H5 시퀀스 + 사전 추출된 Chronos 임베딩"""
    def __init__(self, h5_ds, chronos_embs):
        assert len(h5_ds) == len(chronos_embs)
        self.ds   = h5_ds
        self.embs = chronos_embs.astype(np.float32)

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x_flat, y_ret, y_dir = self.ds[i]
        c_emb = torch.from_numpy(self.embs[i])
        return x_flat, c_emb, y_ret, y_dir


tr_hds = HybridDataset(train_ds, tr_emb)
va_hds = HybridDataset(val_ds,   va_emb)
te_hds = HybridDataset(test_ds,  te_emb)

train_loader = DataLoader(tr_hds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(va_hds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(te_hds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Loaders: train={len(train_loader)}, val={len(val_loader)}, test={len(test_loader)} 배치')

## 5. 학습 & 평가 함수

In [ ]:
def compute_metrics(y_true, y_pred):
    dir_acc  = ((y_true > 0) == (y_pred > 0)).mean()
    spear, _ = spearmanr(y_true, y_pred)
    mae      = np.abs(y_true - y_pred).mean()
    return {
        'dir_acc':  round(float(dir_acc), 4),
        'spearman': round(float(spear), 4),
        'mae':      round(float(mae), 6),
    }


def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total = 0.0
    for x_flat, c_emb, y_ret, _ in loader:
        x_flat, c_emb, y_ret = x_flat.to(DEVICE), c_emb.to(DEVICE), y_ret.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(x_flat, c_emb), y_ret)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    for x_flat, c_emb, y_ret, _ in loader:
        pred = model(x_flat.to(DEVICE), c_emb.to(DEVICE)).cpu().numpy()
        preds.append(pred)
        trues.append(y_ret.numpy())
    return compute_metrics(np.concatenate(trues), np.concatenate(preds))


def train_model(model):
    model     = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)
    criterion = nn.MSELoss()

    best_mae, no_improve, best_state = float('inf'), 0, None
    history = []

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss = train_epoch(model, train_loader, optimizer, criterion)
        val_m   = evaluate(model, val_loader)
        scheduler.step()
        history.append({'epoch': epoch, 'train_loss': tr_loss, **val_m})
        print(f'ep{epoch:02d} | loss={tr_loss:.5f} | val_dir={val_m["dir_acc"]:.4f} | val_mae={val_m["mae"]:.5f}')

        if val_m['mae'] < best_mae:
            best_mae, no_improve = val_m['mae'], 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            no_improve += 1
            if no_improve >= PATIENCE:
                print(f'  Early stop at epoch {epoch}')
                break

    model.load_state_dict(best_state)
    test_m = evaluate(model.to(DEVICE), test_loader)
    print(f'\nTest: {test_m}')
    return model, test_m, history

## 6. 학습 실행

In [ ]:
%%time
gc.collect()
torch.cuda.empty_cache()

hybrid_model, hybrid_metrics, hist = train_model(
    ChronosITransformer(SEQ_LEN, N_FEATURES, CHRONOS_DIM, D_MODEL, N_HEADS, N_LAYERS, DROPOUT)
)
torch.save(hybrid_model.state_dict(), MODEL_DIR / 'hybrid_best.pt')
print('모델 저장 완료: hybrid_best.pt')

## 7. 결과 평가 & 시각화

In [ ]:
# 기존 백본 결과 로드 (비교용)
baseline_results = {
    'StockMixer':          {'dir_acc': 0.5033, 'spearman': 0.0072, 'mae': 0.04515},
    'PatchTST':            {'dir_acc': 0.4960, 'spearman': 0.0096, 'mae': 0.04392},
    'iTransformer':        {'dir_acc': 0.5057, 'spearman': 0.0056, 'mae': 0.04330},
    'Chronos (zero-shot)': {'dir_acc': 0.5096, 'spearman': 0.0192, 'mae': None},
    'Chronos+iTransformer (ours)': hybrid_metrics,
}
results = pd.DataFrame(baseline_results).T
print(results.to_string())

# 학습 곡선
epochs   = [h['epoch'] for h in hist]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(epochs, [h['train_loss'] for h in hist], marker='o', ms=4)
axes[0].set_title('Train Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[1].plot(epochs, [h['dir_acc']    for h in hist], marker='o', ms=4, color='darkorange')
axes[1].axhline(0.5, color='gray', ls='--', lw=1.2, label='Random')
axes[1].set_title('Val Directional Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()
plt.suptitle('ChronosITransformer 학습 곡선', fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / 'hybrid_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

# 모델 비교 막대 그래프
plot_df = results.dropna()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = ['steelblue', 'tomato', 'darkorange', 'seagreen', 'mediumpurple']
for ax, col, title in [
    (axes[0], 'dir_acc',  'Directional Accuracy'),
    (axes[1], 'spearman', 'Spearman Correlation'),
]:
    vals = plot_df[col].astype(float).values
    bars = ax.bar(plot_df.index, vals, color=colors[:len(plot_df)], alpha=0.85, edgecolor='k', lw=0.5)
    if col == 'dir_acc':
        ax.axhline(0.5, color='red', lw=1.2, ls='--', label='Random (50%)')
        ax.legend()
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{v:.4f}', ha='center', fontsize=9)
    ax.set_title(title)
    ax.tick_params(axis='x', rotation=20)
plt.suptitle('백본 모델 비교 (하이브리드 포함)', fontsize=13)
plt.tight_layout()
plt.savefig(DATA_DIR / 'hybrid_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. RL용 임베딩 추출

parquet 전체 데이터에서 슬라이딩 윈도우(lookback=60)로 각 (종목, 날짜)의  
64-dim 임베딩을 추출해 `rl_embeddings.h5`로 저장합니다.  
이 파일을 강화학습 노트북에서 환경 state로 사용합니다.

In [ ]:
FEATURE_COLS = [
    'Adj_Close', 'Open', 'High', 'Low', 'Volume',
    'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
    'MACD', 'MACD_signal', 'MACD_hist',
    'RSI_14', 'BB_upper', 'BB_lower', 'BB_width',
    'Volume_ratio', 'Return_1d', 'Volatility_20d', 'ATR_14',
]  # 20개 — sequences.h5와 동일한 피처 순서

assert len(FEATURE_COLS) == N_FEATURES, f'피처 수 불일치: {len(FEATURE_COLS)}'

print('parquet 로드 중...')
raw_df = pd.read_parquet(PARQUET)
raw_df['Date'] = pd.to_datetime(raw_df['Date'])
raw_df = raw_df.sort_values(['종목코드', 'Date']).reset_index(drop=True)
print(f'parquet: {raw_df.shape}')

In [ ]:
hybrid_model.eval()

EMB_DIM    = 64
LOOKBACK   = SEQ_LEN
BATCH_W    = 256  # 윈도우 배치 크기

# 결과 임시 저장: list of (ticker, date_str, emb)
records = []  # (ticker, date_str, emb_64)

tickers = raw_df['종목코드'].unique()
print(f'총 {len(tickers)}종목 처리 시작...')

for ticker in tqdm(tickers, desc='종목'):
    stock = raw_df[raw_df['종목코드'] == ticker].reset_index(drop=True)
    feat  = stock[FEATURE_COLS].values.astype(np.float32)  # (T, 20)
    dates = stock['Date'].dt.strftime('%Y-%m-%d').values
    T     = len(stock)

    if T < LOOKBACK + 1:
        continue

    # 슬라이딩 윈도우 생성
    windows, window_dates = [], []
    for t in range(LOOKBACK, T):
        window = feat[t - LOOKBACK:t]  # (60, 20)
        if np.isnan(window).any():
            continue
        windows.append(window)
        window_dates.append(dates[t])

    if not windows:
        continue

    windows = np.stack(windows, axis=0)  # (N_win, 60, 20)

    # 배치별 임베딩 추출
    embs = []
    for i in range(0, len(windows), BATCH_W):
        batch_x = torch.tensor(windows[i:i+BATCH_W]).to(DEVICE)        # (B, 60, 20)
        batch_c = extract_chronos_embs(
            windows[i:i+BATCH_W, :, 0], batch_size=BATCH_W, desc=''
        )  # (B, 512) - close price only
        batch_c_t = torch.tensor(batch_c).to(DEVICE)
        emb = hybrid_model.embed(batch_x, batch_c_t).cpu().numpy()      # (B, 64)
        embs.append(emb)

    embs = np.concatenate(embs, axis=0)  # (N_win, 64)
    for date_str, emb in zip(window_dates, embs):
        records.append((ticker, date_str, emb))

print(f'총 {len(records):,}개 (종목, 날짜) 임베딩 추출 완료')

In [ ]:
# rl_embeddings.h5 저장
# 구조: 'tickers'(N), 'dates'(N), 'embeddings'(N, 64)

tickers_arr = np.array([r[0] for r in records], dtype='S10')   # byte string
dates_arr   = np.array([r[1] for r in records], dtype='S10')
embs_arr    = np.stack([r[2] for r in records], axis=0).astype(np.float32)

print(f'저장: tickers={tickers_arr.shape}, dates={dates_arr.shape}, embs={embs_arr.shape}')

with h5py.File(RL_EMB_H5, 'w') as f:
    f.create_dataset('tickers',    data=tickers_arr,  compression='gzip')
    f.create_dataset('dates',      data=dates_arr,    compression='gzip')
    f.create_dataset('embeddings', data=embs_arr,     compression='gzip', chunks=(1000, 64))

print(f'완료: {RL_EMB_H5} ({RL_EMB_H5.stat().st_size/1e6:.1f} MB)')
print()
print('▶ 다음 단계: rl_trading_colab.ipynb 실행')